In [9]:
from pyspark import SparkConf, SparkContext
from pyspark.sql import SparkSession
from pyspark.ml.classification import NaiveBayes, RandomForestClassifier
from pyspark.ml import Pipeline
from pyspark.ml.feature import HashingTF, Tokenizer, IDF
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

In [10]:
# Create a SparkContext
conf = SparkConf().setMaster("local").setAppName("SpamHam_2")

# Stop any existing SparkContext if it's running
if 'SpContext' in globals() and SpContext:
    SpContext.stop()
SpContext = SparkContext(conf = conf)

# Create a SparkSession
SpSession = SparkSession.builder.getOrCreate()

print("Spark Context and Session initialized.")

Spark Context and Session initialized.


In [11]:
!wget -O 10-sms-spam-data.csv https://raw.githubusercontent.com/Luo-Innovation-Lab/Big-Data/main/10-sms-spam-data.csv

--2026-04-12 07:36:46--  https://raw.githubusercontent.com/Luo-Innovation-Lab/Big-Data/main/10-sms-spam-data.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 122231 (119K) [text/plain]
Saving to: ‘10-sms-spam-data.csv’

10-sms-spam-data.cs 100%[===================>] 119.37K  --.-KB/s    in 0.02s   

2026-04-12 07:36:46 (5.72 MB/s) - ‘10-sms-spam-data.csv’ saved [122231/122231]



In [12]:
print("== Start loading data into Spark ==")
# Load the CSV file into an RDD
smsData = SpContext.textFile("10-sms-spam-data.csv", 2)
smsData.cache()

print(f"\nTotal instances loaded: {smsData.count()}")
print("\nFirst 5 rows of raw data:")
for row in smsData.take(5):
    print("\n" + row)

== Start loading data into Spark ==

Total instances loaded: 1000

First 5 rows of raw data:

ham,Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...,,,,,,,,,

ham,Ok lar... Joking wif u oni...,,,,,,,,,,

ham,U dun say so early hor... U c already then say...,,,,,,,,,,

ham,Nah I don't think he goes to usf, he lives around here though,,,,,,,,,

ham,Even my brother is not like to speak with me. They treat me like aids patent.,,,,,,,,,,


In [13]:
print("== Start transforming outcome label data ==")
def TransformToVector(inputStr):
    attList = inputStr.split(",")
    # 0 label for 'ham', 1 for 'spam'
    smsType = 0.0 if attList[0] == "ham" else 1.0
    return [smsType, attList[1]] # spam label, message content

smsXformed = smsData.map(TransformToVector)

smsDf = SpSession.createDataFrame(smsXformed, ["label", "message"])
smsDf.cache()
smsDf.select("label","message").show(5)
print(f"\nTransformed DataFrame count: {smsDf.count()}")

== Start transforming outcome label data ==
+-----+--------------------+
|label|             message|
+-----+--------------------+
|  0.0|Go until jurong p...|
|  0.0|Ok lar... Joking ...|
|  0.0|U dun say so earl...|
|  0.0|Nah I don't think...|
|  0.0|Even my brother i...|
+-----+--------------------+
only showing top 5 rows

Transformed DataFrame count: 1000


In [14]:
print("== Start creating testing and training data ==")
# Split training and testing data (90% training, 10% testing)
(trainingData, testData) = smsDf.randomSplit([0.9, 0.1], seed=42)

print(f"\nTraining data count: {trainingData.count()}")
print(f"\nTest data count: {testData.count()}")

# Display first few rows of test data
print("\nFirst 5 rows of test data:")
testData.show(5)

== Start creating testing and training data ==

Training data count: 911

Test data count: 89

First 5 rows of test data:
+-----+--------------------+
|label|             message|
+-----+--------------------+
|  0.0|1.20 that call co...|
|  0.0|4 oclock at mine....|
|  0.0|Booked ticket for...|
|  0.0|Can you say what ...|
|  0.0|Didn't you get he...|
+-----+--------------------+
only showing top 5 rows


In [15]:
print("== Start building ML pipeline ==")

# Step 1: Tokenize (break sentences into words)
tokenizer = Tokenizer(inputCol="message", outputCol="words")

# Step 2: Calculate Term Frequency (TF)
hashingTF = HashingTF(inputCol=tokenizer.getOutputCol(), numFeatures=2000, outputCol="rawFeatures") # Increased features for better representation

# Step 3: Calculate TF-IDF (Term Frequency-Inverse Document Frequency)
tfIdf = IDF(inputCol=hashingTF.getOutputCol(), outputCol="features")

# Step 4: RandomForest Classifier
rfClassifier = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=100) # Added numTrees parameter

# Build a pipeline, which consists of ML processing stages
pipeline = Pipeline(stages=[tokenizer, hashingTF, tfIdf, rfClassifier])

print("\nML Pipeline constructed.")

== Start building ML pipeline ==

ML Pipeline constructed.


In [16]:
print("== Run ML pipeline ==")
# Train the model with the pipeline
bayesPipeLineModel = pipeline.fit(trainingData) # Training phase

# Make predictions on the test data
prediction = bayesPipeLineModel.transform(testData) # Testing phase

print("\n== Calculate and print evaluation ==")
# Evaluate accuracy
evaluator = MulticlassClassificationEvaluator(
    predictionCol="prediction",
    labelCol="label",
    metricName="accuracy"
)
accuracyResult = evaluator.evaluate(prediction)

# Draw confusion matrix
print("\nConfusion Matrix:")
prediction.groupBy("label","prediction").count().show()

print(f"Accuracy: {accuracyResult:.4f}")
print(f"\nError: {1 - accuracyResult:.4f}")

# Stop Spark Context
SpContext.stop()
print("\nSpark Context stopped.")

== Run ML pipeline ==

== Calculate and print evaluation ==

Confusion Matrix:
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  1.0|       1.0|   38|
|  1.0|       0.0|   12|
|  0.0|       0.0|   39|
+-----+----------+-----+

Accuracy: 0.8652

Error: 0.1348

Spark Context stopped.
